# Update the internal review deck — "IMD_internal_review - full work.pptx"

Packaged from seven scripts, **run in this exact order** against the OneDrive
Presentation folder copy, each one patching or appending to the live file
(never regenerating it from scratch, so earlier hand-edits in PowerPoint
survive). Requires `figs_deck/all_predictions_grid.png` to already exist
(see `build_all_predictions_grid.ipynb`).

**Do not re-run a cell that has already been applied.** The `add_*` cells
append a new slide every time they run — running one twice duplicates that
slide. The `fix_*` / `rebuild_*` cells find-and-replace specific text and
raise if that text is not found (already-changed → will not silently
double-apply, but will error). `reorder_*` moves slides by title text and
is safe to re-run only if the titles have not since changed.

This notebook is a record of how the deck reached its current state, kept so
the steps are legible and reproducible — not meant for routine re-execution.
Close the .pptx in PowerPoint (and let OneDrive finish syncing) before
running any cell here, or the save will fail with a permission error.

## 1. Apply the GHSL-Milan corrections and append the three new-result slides

`build_fullwork_deck.py` — patches the 40-slide reviewed deck (same-source target wording, the 15→20-raster result table, paired-testing scope, the diamonds and bias-recovery slides) and appends the training-label, three-technique and synthesis slides. Source of truth: notebook 05's cross-city CSVs.

In [ ]:
# -*- coding: utf-8 -*-
"""Build "IMD_internal_review - full work.pptx".

Two operations on the reviewed internal deck:

  1. patch the slides that GHS-BUILT-S becoming a Milan training target has made
     wrong or incomplete: the same-source targets slide, the two training-target
     slides, the independent-validation result table (fifteen maps -> twenty),
     the Milan paired-testing slide, the diamonds slide and the bias-recovery
     slide;
  2. append three slides carrying the results computed since the review: the
     four Milan feature sets re-fit on GHS-BUILT-S, the twenty rasters under all
     three validation techniques, and the cross-city synthesis.

Every number is read from notebook 05's cross-city CSVs
(``.../confusion_matrix_validation/cross_city``), which the user has named as
the source of record. Output goes next to the report.

    C:\\ProgramData\\anaconda3\\python.exe build_fullwork_deck.py
"""
import os
import sys

sys.stdout.reconfigure(encoding="utf-8")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from pptx import Presentation
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN
from pptx.util import Emu, Inches, Pt
from scipy.stats import wilcoxon

_PRES = (r"C:\Users\user\OneDrive - Politecnico di Milano\00LCZ\Matej_impervious"
         r"\Ground_truth_validation_S2\Presentation")
SRC = os.path.join(_PRES, "IMD_internal_review.pptx")          # the 40-slide reviewed deck
OUT = os.path.join(_PRES, "IMD_internal_review - full work.pptx")
CSV = (r"C:\Users\user\OneDrive - Politecnico di Milano\File di Daniele Oxoli - PhD_Keerthana"
       r"\test_embeddings\Groundtruth_validation")
CROSS = os.path.join(CSV, "output", "confusion_matrix_validation", "cross_city")
CITY = os.path.join(CSV, "output", "confusion_matrix_validation")
FIGS = "figs_deck"
os.makedirs(FIGS, exist_ok=True)

INK, MUTED, RULE, ACCENT = "#1a1a1a", "#6b6b6b", "#d4d4d4", "#0B6E4F"
GHSL_C, BENCH_C = "#C2724A", "#7d93a3"
TAN, GREY = "#F4E7DF", "#EEF2F4"          # GHSL-trained band / benchmark band
MINUS = "\u2212"
X0, CW = Inches(0.72), Inches(11.9)

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 11,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.grid": True, "grid.color": "#ececec", "grid.linewidth": 0.8,
    "axes.axisbelow": True,
})

A = pd.read_csv(os.path.join(CROSS, "technique_A_continuous_metrics.csv")).set_index(["dataset", "raster"])
B = pd.read_csv(os.path.join(CROSS, "technique_B_hard_confusion_metrics.csv")).set_index(["dataset", "raster"])
C = pd.read_csv(os.path.join(CROSS, "technique_C_level_confusion_metrics.csv")).set_index(["dataset", "raster"])

CHANGES = []


def gv(df, ds, r, c):
    return float(df.loc[(ds, r), c])


def bias_str(v):
    return (MINUS + f"{abs(v):.3f}") if v < 0 else f"{v:.3f}"


# ------------------------------------------------------------------ paired test
def milan_ghsl_wilcoxon():
    """Paired Wilcoxon on per-plot absolute error, each Milan feature set
    CLMS-trained vs the same feature set GHSL-trained. BH within the four."""
    pts = pd.read_csv(os.path.join(CITY, "Milan_2018", "points_extracted.csv"))
    gt = pts["gt_imperv_pct"].to_numpy()
    pairs = [("S2_stack", "S2_stack_GHSL"), ("S2_percentile", "S2_percentile_GHSL"),
             ("S2_median", "S2_median_GHSL"), ("emb_RF", "emb_GHSL")]
    ps = []
    for a, g in pairs:
        ea = np.abs(pts[f"pred_{a}"].to_numpy() - gt)
        eg = np.abs(pts[f"pred_{g}"].to_numpy() - gt)
        m = np.isfinite(ea) & np.isfinite(eg)
        ps.append(wilcoxon(ea[m], eg[m]).pvalue)
    ps = np.array(ps)
    order = ps.argsort()
    q = np.empty_like(ps)
    n = len(ps)
    prev = 1.0
    for rank, idx in enumerate(order[::-1]):
        k = n - rank
        prev = min(prev, ps[idx] * n / k)
        q[idx] = prev
    return float(q.max())


# --------------------------------------------------------------------- figure
def fig_trainlabel():
    groups = ["CLMS-trained\nmodels", "Same features,\nGHSL-trained", "Raw GHS-BUILT-S\nproduct"]
    clms_m = ["S2_stack", "S2_percentile", "S2_median", "emb_RF"]
    ghsl_m = ["S2_stack_GHSL", "S2_percentile_GHSL", "S2_median_GHSL", "emb_GHSL"]

    def band(df, col):
        return [np.mean([gv(df, "Milan_2018", r, col) for r in clms_m]),
                np.mean([gv(df, "Milan_2018", r, col) for r in ghsl_m]),
                gv(df, "Milan_2018", "GHSL", col)]

    fig, axes = plt.subplots(1, 3, figsize=(12.4, 3.8))
    for ax, (col, df, ttl) in zip(axes, [
        ("kappa", B, "Cohen's kappa  (Technique B)"),
        ("quad_weighted_kappa", C, "QWK  (Technique C)"),
        ("bias_pp", A, "Bias, pp  (Technique A)")]):
        vals = band(df, col)
        ax.bar(range(3), vals, color=[ACCENT, GHSL_C, BENCH_C], width=0.62)
        ax.set_xticks(range(3))
        ax.set_xticklabels(groups, fontsize=9)
        ax.set_title(ttl, fontsize=11, fontweight="bold")
        for i, v in enumerate(vals):
            ax.text(i, v, f"{v:.2f}", ha="center",
                    va="bottom" if v >= 0 else "top", fontsize=10)
        if col == "bias_pp":
            ax.axhline(0, color=MUTED, lw=0.8)
    fig.tight_layout()
    p = os.path.join(FIGS, "trainlabel_deck.png")
    fig.savefig(p, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return p


# --------------------------------------------------------------- pptx helpers
def add_slide(prs):
    s = prs.slides.add_slide(prs.slide_layouts[6])
    bg = s.background.fill
    bg.solid()
    bg.fore_color.rgb = RGBColor(0xFF, 0xFF, 0xFF)
    return s


def textbox(slide, x, y, w, h, text, size=12, bold=False, color=INK,
            align=PP_ALIGN.LEFT, spacing=1.25):
    tb = slide.shapes.add_textbox(x, y, w, h)
    tf = tb.text_frame
    tf.word_wrap = True
    for i, line in enumerate(text.split("\n")):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.alignment = align
        p.line_spacing = spacing
        r = p.add_run()
        r.text = line
        r.font.size = Pt(size)
        r.font.bold = bold
        r.font.color.rgb = RGBColor.from_string(color.lstrip("#").upper())
        r.font.name = "Calibri"
    return tb


def rule(slide, y=Inches(2.02)):
    from pptx.enum.shapes import MSO_SHAPE
    sh = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, X0, y, CW, Emu(9525))
    sh.fill.solid()
    sh.fill.fore_color.rgb = RGBColor.from_string(RULE.lstrip("#").upper())
    sh.line.fill.background()
    sh.shadow.inherit = False


def header(slide, kicker, title, sub):
    textbox(slide, X0, Inches(0.30), CW, Inches(0.30), kicker.upper(),
            size=10.5, bold=True, color=ACCENT)
    textbox(slide, X0, Inches(0.60), CW, Inches(0.75), title, size=25, bold=True)
    textbox(slide, X0, Inches(1.42), CW, Inches(0.60), sub, size=13, color=MUTED, spacing=1.3)
    rule(slide)


def footnote(slide, text):
    textbox(slide, X0, Inches(7.03), CW, Inches(0.21), text, size=10.5, color=MUTED, spacing=1.2)


def picture(slide, path, top, max_h, max_w=CW):
    iw, ih = Image.open(path).size
    ar = iw / ih
    w = max_w
    h = Emu(int(w / ar))
    if h > max_h:
        h = max_h
        w = Emu(int(h * ar))
    x = X0 + Emu(int((max_w - w) / 2))
    slide.shapes.add_picture(path, x, top, width=w, height=h)


def build_table(slide, df, x, y, w, h, size=11, band_rows=None):
    rows, cols = df.shape[0] + 1, df.shape[1]
    g = slide.shapes.add_table(rows, cols, x, y, w, h).table
    for j, c in enumerate(df.columns):
        cell = g.cell(0, j)
        cell.text = str(c)
        run = cell.text_frame.paragraphs[0].runs[0]
        run.font.size = Pt(size)
        run.font.bold = True
        run.font.color.rgb = RGBColor(0xFF, 0xFF, 0xFF)
        run.font.name = "Calibri"
        cell.fill.solid()
        cell.fill.fore_color.rgb = RGBColor.from_string(INK.lstrip("#").upper())
    band_rows = band_rows or {}
    for i in range(df.shape[0]):
        tint = band_rows.get(i)
        for j in range(cols):
            cell = g.cell(i + 1, j)
            cell.text = str(df.iat[i, j])
            run = cell.text_frame.paragraphs[0].runs[0]
            run.font.size = Pt(size)
            run.font.name = "Calibri"
            run.font.color.rgb = RGBColor.from_string(INK.lstrip("#").upper())
            cell.fill.solid()
            if tint:
                cell.fill.fore_color.rgb = RGBColor.from_string(tint.lstrip("#").upper())
            else:
                cell.fill.fore_color.rgb = (RGBColor(0xFF, 0xFF, 0xFF) if i % 2 == 0
                                            else RGBColor(0xF7, 0xF7, 0xF7))
    return g


def notes(slide, text):
    try:
        tf = slide.notes_slide.notes_text_frame
        if tf is not None:
            tf.text = text.strip()
    except Exception:
        pass


# ------------------------------------------------------------- patch helpers
def _para_iter(slide):
    for sh in slide.shapes:
        if not sh.has_text_frame:
            continue
        for p in sh.text_frame.paragraphs:
            yield sh, p


def _set_para(p, new_text):
    runs = list(p.runs)
    if not runs:
        r = p.add_run()
        r.text = new_text
        return
    f = runs[0].font
    size, bold, italic, name = f.size, f.bold, f.italic, f.name
    try:
        rgb = f.color.rgb
    except Exception:
        rgb = None
    for r in runs:
        r._r.getparent().remove(r._r)
    r = p.add_run()
    r.text = new_text
    if size is not None:
        r.font.size = size
    r.font.bold = bold
    r.font.italic = italic
    if name:
        r.font.name = name
    if rgb is not None:
        r.font.color.rgb = rgb


def replace_in_slide(slide, sub, new_full, tag):
    for sh, p in _para_iter(slide):
        if sub in "".join(run.text for run in p.runs):
            _set_para(p, new_full)
            CHANGES.append(tag)
            return True
    return False


def replace_notes(slide, old, new, tag):
    try:
        tf = slide.notes_slide.notes_text_frame
    except Exception:
        return False
    if old in tf.text:
        tf.text = tf.text.replace(old, new)
        CHANGES.append(tag)
        return True
    return False


def drop_shape(shape):
    shape._element.getparent().remove(shape._element)


# ------------------------------------------------------------------ 20 maps
MILAN = ["S2_stack", "S2_percentile", "emb_RF", "S2_median", "CLMS",
         "S2_percentile_GHSL", "S2_stack_GHSL", "emb_GHSL", "S2_median_GHSL", "GHSL"]
HANOI = ["emb_localrf", "s2_median_localrf", "s2_median_zeroshot", "emb_zeroshot", "GHSL"]
HCMC = ["s2_median_zeroshot", "emb_localrf", "s2_median_localrf", "GHSL", "emb_zeroshot"]
PRETTY = {
    "S2_stack": "S2 stack", "S2_percentile": "S2 percentile", "S2_median": "S2 median",
    "emb_RF": "emb_RF", "CLMS": "CLMS", "GHSL": "GHS-BUILT-S",
    "S2_stack_GHSL": "S2 stack · GHSL", "S2_percentile_GHSL": "S2 percentile · GHSL",
    "S2_median_GHSL": "S2 median · GHSL", "emb_GHSL": "emb_RF · GHSL",
    "emb_localrf": "emb_localrf", "emb_zeroshot": "emb_zeroshot",
    "s2_median_localrf": "s2_median_localrf", "s2_median_zeroshot": "s2_median_zeroshot",
}


def role(r):
    if r in ("CLMS", "GHSL"):
        return "reference"
    if r.endswith("_GHSL"):
        return "model · GHSL"
    return "model"


def maps_table_df():
    rows, tint = [], {}
    idx = 0
    for city, lst in (("Milan", MILAN), ("Hanoi", HANOI), ("HCMC", HCMC)):
        ds = {"Milan": "Milan_2018", "Hanoi": "Hanoi_2018", "HCMC": "HCMC_2018"}[city]
        for r in lst:
            rows.append([city, PRETTY[r], role(r),
                         f"{gv(A, ds, r, 'RMSE_pp'):.3f}",
                         f"{gv(A, ds, r, 'MAE_pp'):.3f}",
                         bias_str(gv(A, ds, r, "bias_pp"))])
            if city == "Milan" and r.endswith("_GHSL") and r != "GHSL":
                tint[idx] = TAN
            elif r in ("CLMS", "GHSL"):
                tint[idx] = GREY
            idx += 1
    df = pd.DataFrame(rows, columns=["City", "Map", "Role", "RMSE", "MAE", "Bias"])
    return df, tint


# --------------------------------------------------------------------- build
def patch(prs):
    S = list(prs.slides)

    # ---- slide 5 : same-source targets ----------------------------------
    replace_in_slide(
        S[4],
        "Scores each map against the product it was trained on: CLMS in Milan",
        "Scores each map against the product it was trained on: CLMS in Milan, "
        "GHS-BUILT-S in Hanoi and HCMC, and a second Milan model set fitted to "
        "GHS-BUILT-S so the two label choices can be compared on the same features.",
        "S5 same-source targets: Milan now also GHSL")
    replace_in_slide(
        S[4],
        "Runs on the spatial holdout",
        "Runs on the spatial holdout: 1 014 points in Milan, 895 in Hanoi, 887 in HCMC.",
        "S5 holdout line kept")

    # ---- slide 6 : the two training targets ----------------------------
    replace_in_slide(
        S[5], "GHS-BUILT-S \u2014 Hanoi, HCMC",
        "GHS-BUILT-S \u2014 Milan, Hanoi, HCMC", "S6 header: GHSL now spans Milan too")
    replace_in_slide(
        S[5], "an inherent limitation of the chosen reference.",
        "Does not include roads. \u017d" + "gela calls this an inherent limitation of the chosen "
        "reference. Milan was additionally modelled against it, so the effect of the label "
        "choice can be read directly.",
        "S6 GHSL column: note the added Milan run")
    replace_in_slide(
        S[5],
        "This returns later as a number: GHS-BUILT-S under-marks the interpreted reference by close to 20 pp in both cities.",
        "This returns later as a number: against the interpreted reference GHS-BUILT-S "
        "under-marks by close to 20 pp in every city, "
        + f"{MINUS if gv(A,'Milan_2018','GHSL','bias_pp')<0 else ''}"
        + f"+{gv(A,'Milan_2018','GHSL','bias_pp'):.2f} in Milan, +{gv(A,'Hanoi_2018','GHSL','bias_pp'):.2f} "
        f"in Hanoi and +{gv(A,'HCMC_2018','GHSL','bias_pp'):.2f} in HCMC.",
        "S6 footer: three-city bias numbers")

    # ---- slides 25 & 26 : the independent-validation result table ------
    df, tint = maps_table_df()
    for si in (24, 25):
        s = S[si]
        old = next(sh for sh in s.shapes if sh.has_table)
        gx, gy, gw, gh = old.left, old.top, old.width, old.height
        drop_shape(old)
        build_table(s, df, gx, Inches(2.18), gw, Inches(4.62), size=8, band_rows=tint)
        CHANGES.append(f"S{si+1} result table rebuilt: 15 -> 20 maps (notebook 05)")
        replace_in_slide(
            s, "The four-way spread falls from 4.661 RMSE",
            "The four-way spread among the CLMS-trained models falls from 4.661 RMSE "
            "\u2014 33.0 % of the worst map \u2014 to 1.340, or 5.2 %. Most of what the first "
            "validation measured as separation was agreement with CLMS rather than accuracy. "
            "The four GHS-BUILT-S-trained Milan models (tan) sit a clear tier below all five "
            "CLMS-band maps.",
            f"S{si+1} subtitle: name the GHSL-trained tier")
        replace_in_slide(
            s, "All fifteen maps, strict rule",
            "All twenty maps, strict rule, 450 plots per city. Tan rows are the Milan feature "
            "sets fitted to GHS-BUILT-S instead of CLMS.",
            f"S{si+1} footnote: twenty maps")
    # ---- notes housekeeping: fifteen maps -> twenty, everywhere -------
    for si, s in enumerate(prs.slides, 1):
        try:
            tf = s.notes_slide.notes_text_frame
        except Exception:
            tf = None
        if tf is None:
            continue
        t = tf.text
        if not t:
            continue
        n = t
        for a, b in [("worst of all fifteen", "worst model of all twenty"),
                     ("all fifteen at 38.494", "all twenty at 38.494"),
                     ("of all fifteen", "of all twenty"),
                     ("all fifteen maps", "all twenty maps")]:
            n = n.replace(a, b)
        if n != t:
            tf.text = n
            CHANGES.append(f"S{si} notes: fifteen -> twenty")

    # ---- slide 27 : Milan paired testing ----------------------------
    replace_in_slide(
        S[26],
        "What survives is small but real: two tiers of two, not a clean ranking of four",
        "Among the CLMS-trained models, two tiers of two rather than a clean ranking of four",
        "S27 title: scope to CLMS-trained four")
    replace_in_slide(
        S[26],
        "Between tiers q runs from 3.57e-08 to 6.84e-03; neither within-tier pair separates, both at q = 0.79.",
        "Between tiers q runs from 3.57e-08 to 6.84e-03; neither within-tier pair separates, "
        "both at q = 0.79. The four GHS-BUILT-S-trained Milan models are a separate, lower "
        "group (kappa 0.47 to 0.53 against 0.63 to 0.69) and are not part of this within-set test.",
        "S27 footnote: note the GHSL-trained group")

    # ---- slide 29 : the diamonds -----------------------------------
    replace_in_slide(
        S[28],
        "The diamonds mark CLMS and GHS-BUILT-S, scored here as maps under test",
        "The diamonds mark CLMS and GHS-BUILT-S, scored here as maps under test rather than "
        "as targets, on the same 450 plots and the same terms as every model. In Milan "
        "GHS-BUILT-S is now also scored: RMSE 36.811, bias +18.97.",
        "S29 subtitle: Milan GHS-BUILT-S diamond")
    replace_notes(
        S[28],
        "GHS-BUILT-S reaches\n40.345 in Hanoi",
        "GHS-BUILT-S reaches 40.345 in Hanoi", "S29 notes tidy")
    q_ghsl = milan_ghsl_wilcoxon()
    replace_notes(
        S[28],
        "The bias-recovery slide gives the mechanism and measures it.",
        "The bias-recovery slide gives the mechanism and measures it. The same holds for the "
        "Milan GHS-BUILT-S run added since the review: its four models score RMSE 28.2 to 29.3 "
        f"against the product's 36.811, and each is worse than the CLMS-trained model on the "
        f"same features (paired Wilcoxon on per-plot absolute error, every q at or below "
        f"{q_ghsl:.3f}).",
        "S29 notes: Milan GHSL-trained comparison")

    # ---- slide 37 : what retraining recovers -----------------------
    s37 = S[36]
    replace_in_slide(
        s37,
        "GHS-BUILT-S under-marks by 19.68 pp in Hanoi and 19.80 in HCMC.",
        "GHS-BUILT-S under-marks by 18.97 pp in Milan, 19.68 in Hanoi and 19.80 in HCMC.",
        "S37 subtitle: add Milan deficit")
    old = next(sh for sh in s37.shapes if sh.has_table)
    gx, gy, gw, gh = old.left, old.top, old.width, old.height
    drop_shape(old)
    rec = []
    for city, ds, tb, maps in [
        ("Milan", "Milan_2018", gv(A, "Milan_2018", "GHSL", "bias_pp"),
         [("emb_RF · GHSL", "emb_GHSL"), ("S2 stack · GHSL", "S2_stack_GHSL")]),
        ("Hanoi", "Hanoi_2018", 19.68,
         [("emb_localrf", "emb_localrf"), ("S2_median_localrf", "s2_median_localrf")]),
        ("HCMC", "HCMC_2018", 19.80,
         [("emb_localrf", "emb_localrf"), ("S2_median_localrf", "s2_median_localrf")]),
    ]:
        for label, key in maps:
            mb = gv(A, ds, key, "bias_pp")
            rec.append([city, "GHS-BUILT-S", f"{tb:.2f}", label, f"{mb:.2f}",
                        f"{tb - mb:.2f}", f"{100 * (tb - mb) / tb:.1f}"])
    dfr = pd.DataFrame(rec, columns=["City", "Target", "Target bias", "Map", "Map bias",
                                     "Recovered (pp)", "Recovered (%)"])
    build_table(s37, dfr, gx, gy, gw, Inches(3.4), size=10,
                band_rows={0: TAN, 1: TAN})
    CHANGES.append("S37 recovery table: Milan rows added (6 rows)")
    replace_in_slide(
        s37,
        "a target that under-marks by 20 points still yields maps that under-mark by 7 to 12.",
        "a target that under-marks by about 20 points still yields maps that under-mark by 7 to 12; "
        "the Milan GHS-BUILT-S models recover 47 % to 52 %.",
        "S37 footnote: Milan recovery share")



def append_new(prs):
    ftl = fig_trainlabel()
    milan_models = ["S2_stack", "S2_percentile", "S2_median", "emb_RF",
                    "S2_stack_GHSL", "S2_percentile_GHSL", "S2_median_GHSL", "emb_GHSL",
                    "CLMS", "GHSL"]
    tint = {}
    for i, r in enumerate(milan_models):
        if r.endswith("_GHSL") and r != "GHSL":
            tint[i] = TAN
        elif r in ("CLMS", "GHSL"):
            tint[i] = GREY

    q_ghsl = milan_ghsl_wilcoxon()

    # ---- slide A : the training-label result --------------------------
    s = add_slide(prs)
    header(s, "Independent validation \u00b7 the training label",
           "Fit the four Milan feature sets to GHS-BUILT-S and the bias propagates",
           "Milan, same features, same 450 photo-interpreted plots. CLMS labels, then "
           "GHS-BUILT-S labels, then the raw product.")
    picture(s, ftl, Inches(2.25), Inches(3.55))
    textbox(s, X0, Inches(6.00), CW, Inches(1.0),
            "A GHS-BUILT-S-trained model recovers about half of the sealed surface the label "
            "omits, because the Sentinel-2 and embedding features still see roads and yards. "
            "It does not escape the label: Cohen's kappa falls by 0.14 to 0.21 and QWK by "
            "0.09 to 0.11 against the photo reference, and every feature set is worse than its "
            f"CLMS-trained twin (paired Wilcoxon on per-plot absolute error, every q at or "
            f"below {q_ghsl:.3f}).",
            size=12, color=MUTED, spacing=1.3)
    notes(s, "This closes the loop with the label-quality slide. That slide showed the raw "
             "products score no better than the models fitted to them. This shows the "
             "converse: fit a model to the weaker product and the model carries the product's "
             "bias, roughly halved because the features recover some of the omitted surface, "
             "but still a first-order effect that is statistically real. Numbers from notebook "
             "05: Milan GHS-BUILT-S bias +18.97; the four GHSL-trained models +9.19 to +10.02.")

    # ---- slide B : all three techniques, Milan ----------------------
    s = add_slide(prs)
    header(s, "Independent validation \u00b7 three techniques",
           "The ordinal techniques tell the same story as continuous error",
           "Milan, 450 plots, strict rule. Rows grouped: CLMS-trained models, GHS-BUILT-S-"
           "trained models, then the two benchmark products.")
    rowsB = []
    for r in milan_models:
        rowsB.append([PRETTY.get(r, r),
                      f"{gv(A, 'Milan_2018', r, 'RMSE_pp'):.2f}",
                      f"{gv(A, 'Milan_2018', r, 'bias_pp'):+.2f}",
                      f"{gv(B, 'Milan_2018', r, 'kappa'):.3f}",
                      f"{gv(C, 'Milan_2018', r, 'quad_weighted_kappa'):.3f}"])
    dfB = pd.DataFrame(rowsB, columns=["Raster", "RMSE pp (A)", "Bias pp (A)",
                                       "kappa @50% (B)", "QWK (C)"])
    build_table(s, dfB, X0, Inches(2.28), CW, Inches(4.4), size=11, band_rows=tint)
    footnote(s, "Every technique separates the same three bands. Bias is reference minus "
                "predicted; positive means the map under-predicts imperviousness.")
    notes(s, "The internal deck's independent-validation section used continuous error only. "
             "Adding the hard confusion matrix at 50 percent and the threshold-free "
             "fraction-level matrix does not change the ordering: on kappa and on QWK the "
             "CLMS-trained models still lead, the GHS-BUILT-S-trained models sit a clear tier "
             "below, and the raw GHS-BUILT-S product is last.")

    # ---- slide C : cross-city synthesis ---------------------------
    s = add_slide(prs)
    header(s, "Independent validation \u00b7 synthesis",
           "All three techniques pick the same best raster in every city",
           "Ranking each city's rasters under each technique's headline metric. The complete "
           "ranking is identical in Hanoi and HCMC; in Milan the lower ranks reshuffle.")
    best = [("Milan", "Milan_2018", "S2_stack"), ("Hanoi", "Hanoi_2018", "emb_localrf"),
            ("HCMC", "HCMC_2018", "s2_median_zeroshot")]
    rowsC = []
    for city, d, r in best:
        rowsC.append([city, PRETTY.get(r, r),
                      f"{gv(A, d, r, 'RMSE_pp'):.2f}",
                      f"{gv(B, d, r, 'kappa'):.3f}",
                      f"{gv(C, d, r, 'quad_weighted_kappa'):.3f}"])
    dfC = pd.DataFrame(rowsC, columns=["City", "Best raster on all three techniques",
                                       "RMSE pp", "kappa @50%", "QWK"])
    build_table(s, dfC, Inches(1.3), Inches(2.7), Inches(10.7), Inches(1.9), size=13)
    textbox(s, X0, Inches(5.1), CW, Inches(1.4),
            "The three techniques measure different things: absolute error, a binary "
            "decision, and ordinal agreement. They still agree on the top map in each city. "
            "HCMC's winner is a zero-shot map, which is a property of composite depth and "
            "level matching rather than of successful transfer (see the mechanism section).",
            size=12.5, color=MUTED, spacing=1.35)
    notes(s, "This ties the new three-technique validation back to the deck's existing "
             "mechanism argument: the HCMC zero-shot win is consistent across A, B and C, but "
             "the mechanism section already explains it as level matching on a thin "
             "composite, not transfer.")


def main():
    prs = Presentation(SRC)
    patch(prs)
    append_new(prs)
    prs.save(OUT)
    print("saved:", OUT)
    print("slides:", len(prs.slides.__iter__.__self__._sldIdLst))
    print("\nchange log:")
    for c in CHANGES:
        print("  -", c)


if __name__ == "__main__":
    main()

## 2. Add the all-predictions map grid

In [ ]:
# -*- coding: utf-8 -*-
"""Add the all-twenty-predictions map grid to the reviewed internal deck.

Opens "IMD_internal_review - full work.pptx" in place (does not regenerate
it -- build_fullwork_deck.py already applied the GHSL-Milan corrections and
appended the three training-label slides; this only adds one more slide on
top of that, in the deck's own visual style) and appends the grid as a new
slide after the three already-appended new-result slides, so the reviewed
1-43 numbering is undisturbed and this reads as "one more new result."

    C:\\ProgramData\\anaconda3\\python.exe add_all_predictions_to_internal_deck.py
"""
import sys
sys.stdout.reconfigure(encoding="utf-8")

from PIL import Image
from pptx import Presentation
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN
from pptx.util import Emu, Inches, Pt

OUT = (r"C:\Users\user\OneDrive - Politecnico di Milano\00LCZ\Matej_impervious"
       r"\Ground_truth_validation_S2\Presentation\IMD_internal_review - full work.pptx")
IMG = "figs_deck/all_predictions_grid.png"

INK, MUTED, RULE, ACCENT = "#1a1a1a", "#6b6b6b", "#d4d4d4", "#0B6E4F"
X0, CW = Inches(0.72), Inches(11.9)

prs = Presentation(OUT)


def textbox(slide, x, y, w, h, text, size=12, bold=False, color=INK,
            align=PP_ALIGN.LEFT, spacing=1.25):
    tb = slide.shapes.add_textbox(x, y, w, h)
    tf = tb.text_frame
    tf.word_wrap = True
    for i, line in enumerate(text.split("\n")):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.alignment = align
        p.line_spacing = spacing
        r = p.add_run()
        r.text = line
        r.font.size = Pt(size)
        r.font.bold = bold
        r.font.color.rgb = RGBColor.from_string(color.lstrip("#").upper())
        r.font.name = "Calibri"
    return tb


def rule(slide, y=Inches(2.02)):
    from pptx.enum.shapes import MSO_SHAPE
    sh = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, X0, y, CW, Emu(9525))
    sh.fill.solid()
    sh.fill.fore_color.rgb = RGBColor.from_string(RULE.lstrip("#").upper())
    sh.line.fill.background()
    sh.shadow.inherit = False


def header(slide, kicker, title, sub):
    textbox(slide, X0, Inches(0.30), CW, Inches(0.30), kicker.upper(),
            size=10.5, bold=True, color=ACCENT)
    textbox(slide, X0, Inches(0.60), CW, Inches(0.75), title, size=25, bold=True)
    textbox(slide, X0, Inches(1.42), CW, Inches(0.60), sub, size=13, color=MUTED, spacing=1.3)
    rule(slide)


def footnote(slide, text):
    textbox(slide, X0, Inches(7.03), CW, Inches(0.21), text, size=10.5, color=MUTED, spacing=1.2)


def picture(slide, path, top, max_h, max_w=CW):
    iw, ih = Image.open(path).size
    ar = iw / ih
    w = max_w
    h = Emu(int(w / ar))
    if h > max_h:
        h = max_h
        w = Emu(int(h * ar))
    x = X0 + Emu(int((max_w - w) / 2))
    slide.shapes.add_picture(path, x, top, width=w, height=h)


def notes(slide, text):
    try:
        tf = slide.notes_slide.notes_text_frame
        if tf is not None:
            tf.text = text.strip()
    except Exception:
        pass


def add_slide(prs):
    s = prs.slides.add_slide(prs.slide_layouts[6])  # Blank
    bg = s.background.fill
    bg.solid()
    bg.fore_color.rgb = RGBColor(0xFF, 0xFF, 0xFF)
    return s


s = add_slide(prs)
header(s, "Independent validation \u00b7 all predictions",
       "What twenty rasters look like on the ground",
       "Every predicted map in the study, grouped by city and clipped to the same "
       "extent as the rest of its row. Same colour scale throughout, 0\u2013100%.")
picture(s, IMG, Inches(1.85), Inches(4.95))
textbox(s, X0, Inches(6.90), CW, Inches(0.4),
        "Milan: CLMS-trained (row 1) against GHS-BUILT-S-trained (row 2). Hanoi, HCMC: "
        "zero-shot against local retrain, both against the GHS-BUILT-S target.",
        size=11, color=MUTED, spacing=1.2)
notes(s, "The raster counterpart to the RMSE bar chart and the training-label result: "
         "Milan's GHS-BUILT-S-trained row visibly flattens toward the muted GHS-BUILT-S "
         "product itself, and in Vietnam the zero-shot maps are saturated red while the "
         "local retrains recover the target's sparse pattern. GHS-BUILT-S target panels "
         "for Hanoi and HCMC are windowed to the model raster's own extent -- the source "
         "tiles cover a much wider area than the 30 km model AOI.")

prs.save(OUT)
print("saved:", OUT)
print("slides now:", len(prs.slides))

## 3. Move the four new-result slides into the Independent Validation section

In [ ]:
# -*- coding: utf-8 -*-
"""Move the four appended new-result slides into the Independent Validation
section, right after "Robustness" and before "SECTION 6 OF 7". Pure
reordering -- no slide content is touched, so every prior edit survives.

    C:\\ProgramData\\anaconda3\\python.exe reorder_internal_deck.py
"""
import sys
sys.stdout.reconfigure(encoding="utf-8")

from pptx import Presentation

OUT = (r"C:\Users\user\OneDrive - Politecnico di Milano\00LCZ\Matej_impervious"
       r"\Ground_truth_validation_S2\Presentation\IMD_internal_review - full work.pptx")

AFTER_TITLE = "INDEPENDENT VALIDATION \u00b7 ROBUSTNESS"
MOVE_TITLES = [
    "INDEPENDENT VALIDATION \u00b7 THE TRAINING LABEL",
    "INDEPENDENT VALIDATION \u00b7 THREE TECHNIQUES",
    "INDEPENDENT VALIDATION \u00b7 SYNTHESIS",
    "INDEPENDENT VALIDATION \u00b7 ALL PREDICTIONS",
]

prs = Presentation(OUT)


def slide_kicker(slide):
    for sh in slide.shapes:
        if sh.has_text_frame and sh.text_frame.text.strip():
            return sh.text_frame.text.strip().upper()
    return ""


slides = list(prs.slides)
kickers = [slide_kicker(s) for s in slides]

anchor_i = next(i for i, k in enumerate(kickers) if k == AFTER_TITLE)
move_is = [kickers.index(t) for t in MOVE_TITLES]
assert len(set(move_is)) == 4, move_is
print("anchor (Robustness) at position", anchor_i + 1)
print("moving positions", [i + 1 for i in move_is], "->  right after it")

xml_slides = prs.slides._sldIdLst
all_ids = list(xml_slides)
move_ids = [all_ids[i] for i in move_is]
anchor_id = all_ids[anchor_i]

remaining = [sid for sid in all_ids if sid not in move_ids]
insert_at = remaining.index(anchor_id) + 1
new_order = remaining[:insert_at] + move_ids + remaining[insert_at:]

for sid in list(xml_slides):
    xml_slides.remove(sid)
for sid in new_order:
    xml_slides.append(sid)

prs.save(OUT)
print("saved:", OUT)

# ---- verify ---------------------------------------------------------------
prs2 = Presentation(OUT)
for i, s in enumerate(prs2.slides, 1):
    print(i, "|", slide_kicker(s)[:70])

## 4. Rebuild slide 24 ("3 methods"), which had lost its diagram

Replaced four floating, disconnected labels with a proper Technique A/B/C overview panel.

In [ ]:
# -*- coding: utf-8 -*-
"""Rebuild slide 24 ("3 methods"), which had lost its diagram and was left
as four floating, disconnected text labels. Replaced with a proper
three-technique overview panel in the deck's own visual style (matching the
three-column layout already used on the SCOPE and DATA \u00b7 TARGETS slides),
introducing Techniques A/B/C right before the result slides that use them.

Every other slide is untouched.

    C:\\ProgramData\\anaconda3\\python.exe rebuild_slide24.py
"""
import sys
sys.stdout.reconfigure(encoding="utf-8")

from pptx import Presentation
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN
from pptx.util import Emu, Inches, Pt

OUT = (r"C:\Users\user\OneDrive - Politecnico di Milano\00LCZ\Matej_impervious"
       r"\Ground_truth_validation_S2\Presentation\IMD_internal_review - full work.pptx")

INK, MUTED, RULE, ACCENT = "#1a1a1a", "#6b6b6b", "#d4d4d4", "#0B6E4F"
X0, CW = Inches(0.72), Inches(11.9)
COL_X = [Inches(0.72), Inches(4.80), Inches(8.89)]
COL_W = Inches(3.73)

prs = Presentation(OUT)


def textbox(slide, x, y, w, h, text, size=12, bold=False, color=INK,
            align=PP_ALIGN.LEFT, spacing=1.25):
    tb = slide.shapes.add_textbox(x, y, w, h)
    tf = tb.text_frame
    tf.word_wrap = True
    for i, line in enumerate(text.split("\n")):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.alignment = align
        p.line_spacing = spacing
        r = p.add_run()
        r.text = line
        r.font.size = Pt(size)
        r.font.bold = bold
        r.font.color.rgb = RGBColor.from_string(color.lstrip("#").upper())
        r.font.name = "Calibri"
    return tb


def rule(slide, y=Inches(2.02)):
    from pptx.enum.shapes import MSO_SHAPE
    sh = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, X0, y, CW, Emu(9525))
    sh.fill.solid()
    sh.fill.fore_color.rgb = RGBColor.from_string(RULE.lstrip("#").upper())
    sh.line.fill.background()
    sh.shadow.inherit = False


def header(slide, kicker, title, sub):
    textbox(slide, X0, Inches(0.30), CW, Inches(0.30), kicker.upper(),
            size=10.5, bold=True, color=ACCENT)
    textbox(slide, X0, Inches(0.60), CW, Inches(0.75), title, size=25, bold=True)
    textbox(slide, X0, Inches(1.42), CW, Inches(0.60), sub, size=13, color=MUTED, spacing=1.3)
    rule(slide)


def footnote(slide, text):
    textbox(slide, X0, Inches(7.03), CW, Inches(0.21), text, size=10.5, color=MUTED, spacing=1.2)


def notes(slide, text):
    try:
        tf = slide.notes_slide.notes_text_frame
        if tf is not None:
            tf.text = text.strip()
    except Exception:
        pass


def column(slide, x, heading, lines):
    textbox(slide, x, Inches(2.34), COL_W, Inches(0.45), heading, size=13, bold=True, color=INK)
    textbox(slide, x, Inches(2.79), COL_W, Inches(4.1), "\n".join(lines),
            size=12, color=INK, spacing=1.3)


def find_slide(kicker_text):
    for s in prs.slides:
        for sh in s.shapes:
            if sh.has_text_frame and sh.text_frame.text.strip().upper() == kicker_text:
                return s
    raise KeyError(kicker_text)


s = find_slide("3 METHODS")
for sh in list(s.shapes):
    sh._element.getparent().remove(sh._element)

header(s, "Independent validation \u00b7 method",
       "Three techniques, the same 450 plots, never merged",
       "Each of the twenty rasters is scored three times against the photo-interpreted "
       "reference. The result slides that follow report all three.")

column(s, COL_X[0], "A \u00b7 Continuous",
       ["Raw predicted % against the reference %, no threshold.",
        "RMSE, MAE, bias and R\u00b2, in IMD percentage points.",
        "Bias is observed minus predicted: positive means the map under-predicts."])
column(s, COL_X[1], "B \u00b7 Hard confusion",
       ["Both map and reference cut to pervious / impervious at a 50% threshold.",
        "Overall accuracy and Cohen's kappa, the headline number.",
        "Per-class precision, recall and F1 for the impervious class."])
column(s, COL_X[2], "C \u00b7 10-class ordinal",
       ["Map and reference both binned into the same 10 IMD levels, no threshold.",
        "Exact-match and within-one-level accuracy.",
        "Quadratic-weighted kappa (QWK), the headline number \u2014 partial credit for "
        "being one level off, none for being further."])

footnote(s, "A, B and C measure different things (error, a binary decision, ordinal "
            "agreement) and can disagree about which map is better \u2014 the result "
            "slides report where they agree and where they do not.")
notes(s, "This replaces the original slide, which had lost its diagram at some point and "
         "was left as four disconnected text labels (\"3 methods / Continues / Binary / "
         "10 class\") with no title or explanation. Rebuilt as a proper technique overview "
         "using the same three-column layout as the SCOPE and DATA \u00b7 TARGETS slides. "
         "Technique definitions match imd_confusion_matrix_validation.ipynb and the "
         "ground-truth validation report.")

prs.save(OUT)
print("saved:", OUT)

## 5. Fix the same-source holdout line on slide 5

"1 014 points in Milan" was the CLMS-trained holdout only; the GHSL-trained models run on a separate, independently resampled 998-point holdout (notebook 01c).

In [ ]:
# -*- coding: utf-8 -*-
"""Fix slide 5 ("METHOD · THE HINGE"): the same-source holdout line quoted a
single Milan count, 1 014, but that is specifically the CLMS-trained holdout
(Zgela's original 3 500-point sample, 1 km blocks, 250 m buffer). The four
GHS-BUILT-S-trained Milan models run on a different, independently resampled
3 500-point set (Matej's-method stratified draw from GHSL's own
classification), split the same way: 2 450 train / 998 test after the 250 m
buffer removes 52 points (notebook 01c, cell 11 output). That is a different
number and was missing.

    C:\\ProgramData\\anaconda3\\python.exe fix_slide5_holdout.py
"""
import sys
sys.stdout.reconfigure(encoding="utf-8")

from pptx import Presentation

OUT = (r"C:\Users\user\OneDrive - Politecnico di Milano\00LCZ\Matej_impervious"
       r"\Ground_truth_validation_S2\Presentation\IMD_internal_review - full work.pptx")

OLD = "Runs on the spatial holdout: 1 014 points in Milan, 895 in Hanoi, 887 in HCMC."
NEW = ("Runs on the spatial holdout: 1 014 points in Milan against CLMS, and a separate "
       "998-point holdout against GHS-BUILT-S drawn from an independent resample of "
       "GHS-BUILT-S's own classification; 895 in Hanoi, 887 in HCMC.")

prs = Presentation(OUT)


def set_para(p, new_text):
    runs = list(p.runs)
    if not runs:
        r = p.add_run()
        r.text = new_text
        return
    f = runs[0].font
    size, bold, italic, name = f.size, f.bold, f.italic, f.name
    try:
        rgb = f.color.rgb
    except Exception:
        rgb = None
    for r in runs:
        r._r.getparent().remove(r._r)
    r = p.add_run()
    r.text = new_text
    if size is not None:
        r.font.size = size
    r.font.bold = bold
    r.font.italic = italic
    if name:
        r.font.name = name
    if rgb is not None:
        r.font.color.rgb = rgb


found = False
for s in prs.slides:
    for sh in s.shapes:
        if not sh.has_text_frame:
            continue
        for p in sh.text_frame.paragraphs:
            if OLD in "".join(r.text for r in p.runs):
                set_para(p, NEW)
                found = True

if not found:
    raise SystemExit("OLD text not found -- slide 5 wording may already differ; aborting "
                      "without saving so nothing is clobbered blind.")

prs.save(OUT)
print("patched slide 5 holdout line, saved:", OUT)

## 6. Add the Milan GHS-BUILT-S same-source results table

The deck referenced a "second Milan model set fitted to GHS-BUILT-S" (slide 5) but never showed its same-source numbers on their own terms. Mirrors the CLMS table; scored on the 998-point holdout.

In [ ]:
# -*- coding: utf-8 -*-
"""Add the Milan GHS-BUILT-S same-source results table -- the result slide 5
("METHOD \u00b7 THE HINGE") promises ("a second Milan model set fitted to
GHS-BUILT-S") but the deck never actually showed. Mirrors slide 13's CLMS
table exactly, scored on the model's own 998-point holdout (notebook 01c/01d,
holdout_test_metrics.csv in each outputs_*_GHSL directory -- GEE_RF row).

Inserted right after "MILAN \u00b7 MECHANISM" and before "SECTION 4 OF 7", closing
out the Milan section with the GHS-BUILT-S same-source table before Vietnam
starts (Vietnam is scored against GHS-BUILT-S throughout, so this bridges).

Finding worth surfacing: against GHS-BUILT-S the four feature sets nearly
converge (0.58 RMSE spread, 3.1% of the worst map) -- a much smaller spread
than the 4.66 RMSE / 33.0% spread against CLMS on slide 13. The label being
noisier compresses the composite-choice advantage even in-sample, before the
independent-validation compression story later in the deck.

Every other slide is untouched.

    C:\\ProgramData\\anaconda3\\python.exe add_milan_ghsl_samesource_slide.py
"""
import sys
sys.stdout.reconfigure(encoding="utf-8")

from pptx import Presentation
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN
from pptx.util import Emu, Inches, Pt

OUT = (r"C:\Users\user\OneDrive - Politecnico di Milano\00LCZ\Matej_impervious"
       r"\Ground_truth_validation_S2\Presentation\IMD_internal_review - full work.pptx")

INK, MUTED, RULE, ACCENT = "#1a1a1a", "#6b6b6b", "#d4d4d4", "#0B6E4F"
X0, CW = Inches(0.72), Inches(11.9)

# GEE_RF holdout_test_metrics.csv, 998-point holdout, one row per Milan
# GHSL-trained feature set (verified against the run directories directly).
ROWS = [  # name, RMSE, MAE, R2, Bias
    ("S2 percentile (50 bands)",       18.206, 13.523, 0.737,  0.341),
    ("S2 stack (40 bands)",            18.288, 13.793, 0.734, -0.140),
    ("AlphaEarth embeddings (64 bands)", 18.361, 13.820, 0.732, 0.339),
    ("S2 median (10 bands)",           18.785, 14.125, 0.720,  0.386),
]
# Same order, SVR:
SVR_ROWS = [18.431, 18.499, 19.217, 18.910]

prs = Presentation(OUT)


def textbox(slide, x, y, w, h, text, size=12, bold=False, color=INK,
            align=PP_ALIGN.LEFT, spacing=1.25):
    tb = slide.shapes.add_textbox(x, y, w, h)
    tf = tb.text_frame
    tf.word_wrap = True
    for i, line in enumerate(text.split("\n")):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.alignment = align
        p.line_spacing = spacing
        r = p.add_run()
        r.text = line
        r.font.size = Pt(size)
        r.font.bold = bold
        r.font.color.rgb = RGBColor.from_string(color.lstrip("#").upper())
        r.font.name = "Calibri"
    return tb


def rule(slide, y=Inches(2.02)):
    from pptx.enum.shapes import MSO_SHAPE
    sh = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, X0, y, CW, Emu(9525))
    sh.fill.solid()
    sh.fill.fore_color.rgb = RGBColor.from_string(RULE.lstrip("#").upper())
    sh.line.fill.background()
    sh.shadow.inherit = False


def header(slide, kicker, title, sub):
    textbox(slide, X0, Inches(0.30), CW, Inches(0.30), kicker.upper(),
            size=10.5, bold=True, color=ACCENT)
    textbox(slide, X0, Inches(0.60), CW, Inches(0.75), title, size=25, bold=True)
    textbox(slide, X0, Inches(1.42), CW, Inches(0.60), sub, size=13, color=MUTED, spacing=1.3)
    rule(slide)


def footnote(slide, text):
    textbox(slide, X0, Inches(7.03), CW, Inches(0.21), text, size=10.5, color=MUTED, spacing=1.2)


def notes(slide, text):
    try:
        tf = slide.notes_slide.notes_text_frame
        if tf is not None:
            tf.text = text.strip()
    except Exception:
        pass


def build_table(slide, headers, rows, x, y, w, h, size=13):
    g = slide.shapes.add_table(len(rows) + 1, len(headers), x, y, w, h).table
    for j, hd in enumerate(headers):
        cell = g.cell(0, j)
        cell.text = str(hd)
        run = cell.text_frame.paragraphs[0].runs[0]
        run.font.size = Pt(size)
        run.font.bold = True
        run.font.color.rgb = RGBColor(0xFF, 0xFF, 0xFF)
        run.font.name = "Calibri"
        cell.fill.solid()
        cell.fill.fore_color.rgb = RGBColor.from_string(INK.lstrip("#").upper())
    for i, row in enumerate(rows):
        for j, v in enumerate(row):
            cell = g.cell(i + 1, j)
            cell.text = str(v)
            run = cell.text_frame.paragraphs[0].runs[0]
            run.font.size = Pt(size)
            run.font.name = "Calibri"
            run.font.color.rgb = RGBColor.from_string(INK.lstrip("#").upper())
            cell.fill.solid()
            cell.fill.fore_color.rgb = (RGBColor(0xFF, 0xFF, 0xFF) if i % 2 == 0
                                        else RGBColor(0xF7, 0xF7, 0xF7))
    return g


def add_slide(prs):
    s = prs.slides.add_slide(prs.slide_layouts[6])  # Blank
    bg = s.background.fill
    bg.solid()
    bg.fore_color.rgb = RGBColor(0xFF, 0xFF, 0xFF)
    return s


def bias_str(v):
    return f"{v:+.3f}"


# ---- build the new slide ---------------------------------------------------
spread_ghsl = max(r[1] for r in ROWS) - min(r[1] for r in ROWS)
worst_ghsl = max(r[1] for r in ROWS)
pct_ghsl = 100 * spread_ghsl / worst_ghsl

s = add_slide(prs)
header(s, "Milan \u00b7 same-source (GHS-BUILT-S)",
       "Against GHS-BUILT-S, the four feature sets nearly converge",
       "The same four Milan feature sets, refit on GHS-BUILT-S labels instead of CLMS, "
       "scored on their own 998-point spatial holdout (an independent resample of "
       "GHS-BUILT-S's own classification, split the same way as the CLMS design).")

headers = ["Predictor set", "RMSE", "MAE", "R\u00b2", "Bias"]
rows = [[name, f"{rmse:.3f}", f"{mae:.3f}", f"{r2:.3f}", bias_str(bias)]
        for name, rmse, mae, r2, bias in ROWS]
build_table(s, headers, rows, X0, Inches(2.34), CW, Inches(2.55), size=14)

textbox(s, X0, Inches(5.25), CW, Inches(1.55),
        f"The spread is {spread_ghsl:.3f} RMSE, {pct_ghsl:.1f}% of the worst map \u2014 against "
        f"4.661 RMSE, 33.0% of the worst map, on the equivalent CLMS-trained table. GHS-BUILT-S "
        "is the noisier label: which features you feed it matters far less than it does against "
        "CLMS, even before any independent check. That compression is real here, in-sample, not "
        "only against the 450 photo-interpreted plots later in this deck.",
        size=12.5, color=MUTED, spacing=1.35)

footnote(s, "GEE random forest, 998-point spatial holdout, scored against GHS-BUILT-S. The SVR "
            "rows run in the same order: "
            + " / ".join(f"{v:.3f}" for v in SVR_ROWS) + ".")

notes(s, "This is the same-source counterpart to slide 13 (Milan vs CLMS), for the models "
         "trained on GHS-BUILT-S instead -- a result the deck referenced (slide 5, 'a second "
         "Milan model set fitted to GHS-BUILT-S') but never actually showed on its own terms. "
         "Numbers from outputs_GHSL/holdout_test_metrics.csv, outputs_S2_stack_GHSL/..., "
         "outputs_S2_median_GHSL/..., outputs_S2_percentile_p10p25p50p75p90_GHSL/... (GEE_RF "
         "row of each). Placed at the end of the Milan section, right before Vietnam, because "
         "Vietnam is scored against GHS-BUILT-S throughout -- this bridges the two.\n\n"
         "The convergence here is a genuinely different result from the independent-validation "
         "compression (slide 25-26/32): that compression is same predictors, two different "
         "references (CLMS vs 450 plots). This one is the same reference (GHS-BUILT-S), four "
         "different predictors -- the spread shrinks because the label itself is noisier, not "
         "because the reference changed. Keep the two distinct if asked.")

# ---- reorder: insert right after "MILAN · MECHANISM" ----------------------
def slide_kicker(slide):
    for sh in slide.shapes:
        if sh.has_text_frame and sh.text_frame.text.strip():
            return sh.text_frame.text.strip().upper()
    return ""


xml_slides = prs.slides._sldIdLst
all_ids = list(xml_slides)
new_id = all_ids[-1]
all_ids = all_ids[:-1]

kickers = [slide_kicker(sl) for sl in list(prs.slides)[:-1]]
anchor_i = kickers.index("MILAN \u00b7 MECHANISM")
new_order = all_ids[:anchor_i + 1] + [new_id] + all_ids[anchor_i + 1:]

for sid in list(xml_slides):
    xml_slides.remove(sid)
for sid in new_order:
    xml_slides.append(sid)

prs.save(OUT)
print("saved:", OUT)

prs2 = Presentation(OUT)
print("total slides:", len(prs2.slides))
for i, sl in enumerate(prs2.slides, 1):
    print(i, "|", slide_kicker(sl)[:70])

## 7. Add the GHS-BUILT-S Milan spatial train/test split figure

Companion to the CLMS split slide — the GHSL-trained models use a different, independently resampled point set, split the same way (1 km blocks, 250 m buffer).

In [ ]:
# -*- coding: utf-8 -*-
"""Add the GHS-BUILT-S Milan spatial train/test split figure, the companion
to slide 10 ("METHOD \u00b7 SAMPLING") which shows only the CLMS split.

The GHSL-trained Milan models use a different, independently resampled
3 500-point set (Matej's-method stratified draw from GHS-BUILT-S's own
classification, 500/class), split by the same 1 km blocks / 250 m buffer
design -- notebook 01c, fig01_spatial_split.png: 2 450 train / 998 test / 52
removed by buffer. That figure existed on disk but was never in the deck.

Inserted right after "METHOD \u00b7 SAMPLING" (the CLMS split), so the two sit
side by side as a pair. Every other slide is untouched.

    C:\\ProgramData\\anaconda3\\python.exe add_milan_ghsl_split_slide.py
"""
import sys
sys.stdout.reconfigure(encoding="utf-8")

from PIL import Image
from pptx import Presentation
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN
from pptx.util import Emu, Inches, Pt

OUT = (r"C:\Users\user\OneDrive - Politecnico di Milano\00LCZ\Matej_impervious"
       r"\Ground_truth_validation_S2\Presentation\IMD_internal_review - full work.pptx")
IMG = "figs_deck/milan_ghsl_spatial_split.png"
ANCHOR_KICKER = "METHOD \u00b7 SAMPLING"

INK, MUTED, RULE, ACCENT = "#1a1a1a", "#6b6b6b", "#d4d4d4", "#0B6E4F"
X0, CW = Inches(0.72), Inches(11.9)

prs = Presentation(OUT)


def textbox(slide, x, y, w, h, text, size=12, bold=False, color=INK,
            align=PP_ALIGN.LEFT, spacing=1.25):
    tb = slide.shapes.add_textbox(x, y, w, h)
    tf = tb.text_frame
    tf.word_wrap = True
    for i, line in enumerate(text.split("\n")):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.alignment = align
        p.line_spacing = spacing
        r = p.add_run()
        r.text = line
        r.font.size = Pt(size)
        r.font.bold = bold
        r.font.color.rgb = RGBColor.from_string(color.lstrip("#").upper())
        r.font.name = "Calibri"
    return tb


def rule(slide, y=Inches(2.02)):
    from pptx.enum.shapes import MSO_SHAPE
    sh = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, X0, y, CW, Emu(9525))
    sh.fill.solid()
    sh.fill.fore_color.rgb = RGBColor.from_string(RULE.lstrip("#").upper())
    sh.line.fill.background()
    sh.shadow.inherit = False


def header(slide, kicker, title, sub):
    textbox(slide, X0, Inches(0.30), CW, Inches(0.30), kicker.upper(),
            size=10.5, bold=True, color=ACCENT)
    textbox(slide, X0, Inches(0.60), CW, Inches(0.75), title, size=25, bold=True)
    textbox(slide, X0, Inches(1.42), CW, Inches(0.60), sub, size=13, color=MUTED, spacing=1.3)
    rule(slide)


def footnote(slide, text):
    textbox(slide, X0, Inches(7.03), CW, Inches(0.21), text, size=10.5, color=MUTED, spacing=1.2)


def picture(slide, path, top, max_h, max_w=CW):
    iw, ih = Image.open(path).size
    ar = iw / ih
    w = max_w
    h = Emu(int(w / ar))
    if h > max_h:
        h = max_h
        w = Emu(int(h * ar))
    x = X0 + Emu(int((max_w - w) / 2))
    slide.shapes.add_picture(path, x, top, width=w, height=h)


def notes(slide, text):
    try:
        tf = slide.notes_slide.notes_text_frame
        if tf is not None:
            tf.text = text.strip()
    except Exception:
        pass


def add_slide(prs):
    s = prs.slides.add_slide(prs.slide_layouts[6])
    bg = s.background.fill
    bg.solid()
    bg.fore_color.rgb = RGBColor(0xFF, 0xFF, 0xFF)
    return s


def slide_kicker(slide):
    for sh in slide.shapes:
        if sh.has_text_frame and sh.text_frame.text.strip():
            return sh.text_frame.text.strip().upper()
    return ""


# ---- build the new slide ---------------------------------------------------
s = add_slide(prs)
header(s, "Method \u00b7 sampling (GHS-BUILT-S)",
       "The GHS-BUILT-S resample is split the same way",
       "An independent 3 500-point stratified resample of GHS-BUILT-S's own classification "
       "(500 per class, Matej's method), split by the same 1 km blocks and 250 m buffer as "
       "the CLMS design on the previous slide.")
picture(s, IMG, Inches(2.22), Inches(4.0))
footnote(s, "2 450 training and 998 test points, roughly 71/29, after the 250 m buffer removes "
            "52 test points. Class balance across the 7 GHS-BUILT-S classes, train vs test.")
notes(s, "Companion to the CLMS split on the previous slide. The two point sets are different "
         "by construction: CLMS training points are Zgela's original 3 500-point stratified "
         "sample of the CLMS classification; these are an independently drawn 3 500-point "
         "stratified sample of GHS-BUILT-S's own classification (same 500-per-class, 7-class "
         "design, same 1 km block / 250 m buffer split), used to train and test the four "
         "GHS-BUILT-S-labelled Milan models. That is why the same-source holdout size differs "
         "(1 014 vs 998) even though the design is otherwise identical -- see slide 5's "
         "same-source line and the GHS-BUILT-S same-source table at the end of this section. "
         "Figure from notebook 01c (outputs_GHSL/fig01_spatial_split.png).")

# ---- reorder: insert right after "METHOD · SAMPLING" -----------------------
xml_slides = prs.slides._sldIdLst
all_ids = list(xml_slides)
new_id = all_ids[-1]
all_ids = all_ids[:-1]

kickers = [slide_kicker(sl) for sl in list(prs.slides)[:-1]]
anchor_i = kickers.index(ANCHOR_KICKER)
new_order = all_ids[:anchor_i + 1] + [new_id] + all_ids[anchor_i + 1:]

for sid in list(xml_slides):
    xml_slides.remove(sid)
for sid in new_order:
    xml_slides.append(sid)

prs.save(OUT)
print("saved:", OUT)

prs2 = Presentation(OUT)
print("total slides:", len(prs2.slides))
for i, sl in enumerate(prs2.slides, 1):
    print(i, "|", slide_kicker(sl)[:70])

## Superseded: `add_trainlabel_slides.py`

An earlier version of step 1 above, run once against the *old* 38-slide
repo-root copy of the deck before the reviewed 40-slide OneDrive version was
identified as the correct source. Its output was fully superseded by step 1
(`build_fullwork_deck.py`, run against the correct source) and is not part of
the current deck's history. Kept out of this notebook; the script itself was
deleted from the repo.